# AIOps Week 2 - Day 2: Root Cause Analysis (RCA) Pipeline
This notebook implements a step-by-step Root Cause Analysis (RCA) pipeline for the GeekShop platform. The system combines graph centrality (PageRank), temporal analysis, and TF-IDF similarity matching against historical incident logs. It also visualizes the service topology and cluster subgraphs.

In [1]:
# Step 1: Import packages and define dataset paths
import json
import os
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Define paths
dataset_dir = "./dataset"
cluster_summary_path = "../d1/results/cluster_summary.json"
services_path = os.path.join(dataset_dir, "services.json")
alerts_path = os.path.join(dataset_dir, "alerts_sample.jsonl")
history_path = os.path.join(dataset_dir, "incidents_history.json")
output_path = "./results/rca_output.json"

In [2]:
# Step 2: Load datasets
with open(cluster_summary_path, encoding='utf-8') as f:
    cluster_summary = json.load(f)

with open(services_path, encoding='utf-8') as f:
    services_data = json.load(f)

with open(history_path, encoding='utf-8') as f:
    history_data = json.load(f)

all_alerts = []
with open(alerts_path, encoding='utf-8') as f:
    for line in f:
        if line.strip():
            all_alerts.append(json.loads(line))

print("Successfully loaded:")
print(f"  - {len(cluster_summary['clusters'])} alert clusters from Day 1")
print(f"  - {len(services_data['services'])} services and {len(services_data['stores'])} stores from topology")
print(f"  - {len(all_alerts)} raw alerts")
print(f"  - {len(history_data['incidents'])} historical incidents")

Successfully loaded:
  - 6 alert clusters from Day 1
  - 10 services and 4 stores from topology
  - 20 raw alerts
  - 29 historical incidents


In [3]:
# Step 3: Build Topology Graph, define alert filtering helper, and visualize
G = nx.DiGraph()
for svc in services_data['services']:
    G.add_node(svc['name'], type='service', criticality=svc.get('criticality', 'medium'))
for store in services_data['stores']:
    G.add_node(store['name'], type='store', criticality=store.get('criticality', 'medium'))
for edge in services_data['edges']:
    G.add_edge(edge['from'], edge['to'], type=edge.get('type', 'http'))

stores = services_data['stores']

def get_alerts_for_cluster(cluster, alerts):
    """Filter raw alerts that belong to a cluster by matching fingerprint, service, and time range."""
    cluster_services = set(cluster["services"])
    cluster_fingerprints = set(cluster["fingerprints"])
    t_start = pd.to_datetime(cluster["time_range"][0])
    t_end = pd.to_datetime(cluster["time_range"][1])
    
    matched = []
    for a in alerts:
        a_fp = f"{a['service']}|{a['metric']}|{a['severity']}"
        a_ts = pd.to_datetime(a["ts"])
        if a["service"] in cluster_services and a_fp in cluster_fingerprints and t_start <= a_ts <= t_end:
            matched.append(a)
    return matched

print(f"Service graph built with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")

# Draw the topology with a clean hierarchical layout
plt.figure(figsize=(15, 10))
pos = {
    'edge-lb': (0.5, 3.0),
    'auth-svc': (0.05, 2.0),
    'catalog-svc': (0.2, 2.0),
    'search-svc': (0.35, 2.0),
    'checkout-svc': (0.5, 2.0),
    'cart-svc': (0.65, 2.0),
    'payment-svc': (0.80, 2.0),
    'inventory-svc': (0.95, 2.0),
    'notification-svc': (0.75, 1.2),
    'recommender-svc': (0.2, 1.2),
    'cart-redis': (0.65, 0.4),
    'catalog-db': (0.3, 0.4),
    'payments-db': (0.8, 0.4),
    'kafka-events': (0.5, 0.4)
}

# Handle any missing nodes safely
for node in G.nodes():
    if node not in pos:
        pos[node] = (0.5, 1.0)

# Offset labels slightly upwards so they never overlap with the node circles
pos_labels = {k: (v[0], v[1] + 0.18) for k, v in pos.items()}

node_colors = []
for node, attrs in G.nodes(data=True):
    if attrs.get('type') == 'store':
        node_colors.append('#ff7979')  # Soft pastel red
    else:
        node_colors.append('#54a0ff')  # Soft pastel blue

nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=1200, 
                       edgecolors='#2c3e50', linewidths=1.5)
nx.draw_networkx_labels(G, pos_labels, font_size=9, font_weight='bold')
nx.draw_networkx_edges(G, pos, arrowstyle="->", arrowsize=18, 
                       edge_color="#7f8c8d", width=1.5, alpha=0.7, 
                       connectionstyle="arc3,rad=0.1")

plt.title("GeekShop Premium Service Topology Architecture Map\n(Blue = Application Services, Red = DB/Cache Stores)", 
          fontsize=14, fontweight='bold', pad=25)

# Add padding to axes to prevent cropping
ax = plt.gca()
ax.set_xlim(-0.1, 1.1)
ax.set_ylim(0.1, 3.4)

plt.axis('off')
plt.tight_layout()
plt.show()

Service graph built with 14 nodes and 17 edges.


In [4]:
# Step 4: Initialize TF-IDF Vectorizer for historical incident logs
incident_list = history_data["incidents"]
documents = []
for inc in incident_list:
    svcs = " ".join(inc["services_involved"])
    doc = f"{svcs} {inc['root_cause_service']} {inc['summary']}"
    documents.append(doc)

vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform(documents)

print(f"TF-IDF matrix computed for {len(documents)} historical incidents. Vocabulary size: {len(vectorizer.vocabulary_)} words.")

TF-IDF matrix computed for 29 historical incidents. Vocabulary size: 198 words.


In [5]:
# Step 5: Implement Graph + Temporal Scorer with Terminal Noise Adjustment
def compute_graph_temporal_candidates(cluster, c_alerts):
    services = cluster["services"]
    
    # A. PageRank on reversed subgraph
    subg = G.subgraph(services)
    if len(services) > 1 and subg.number_of_edges() > 0:
        pagerank_scores = nx.pagerank(subg.reverse(copy=True), alpha=0.85)
    else:
        pagerank_scores = {node: 1.0 / len(services) for node in services}
        
    max_pr = max(pagerank_scores.values()) if pagerank_scores else 1.0
    pagerank_norm = {node: (score / max_pr if max_pr > 0 else 1.0) for node, score in pagerank_scores.items()}
    
    # B. Temporal Scorer based on earliest alert timestamp
    earliest_ts = {}
    for s in services:
        s_alerts = [a for a in c_alerts if a["service"] == s]
        if s_alerts:
            earliest_ts[s] = min(pd.to_datetime(a["ts"]) for a in s_alerts)
        else:
            earliest_ts[s] = pd.to_datetime(cluster["time_range"][0])
            
    ts_values = list(earliest_ts.values())
    t_min = min(ts_values)
    t_max = max(ts_values)
    
    timestamp_score = {}
    if t_max == t_min:
        for s in services:
            timestamp_score[s] = 1.0
    else:
        range_sec = (t_max - t_min).total_seconds()
        for s in services:
            ts = earliest_ts[s]
            diff_sec = (ts - t_min).total_seconds()
            timestamp_score[s] = 1.0 - (diff_sec / range_sec if range_sec > 0 else 0)
            
    # C. Combined score (0.6 * PageRank + 0.4 * Timestamp)
    combined_score = {}
    for s in services:
        combined_score[s] = 0.6 * pagerank_norm[s] + 0.4 * timestamp_score[s]
        
    # Sort candidates
    sorted_candidates = sorted(combined_score.items(), key=lambda x: x[1], reverse=True)
    
    # D. Terminal noise adjustment: nếu top candidate là DB/cache (store),
    #    kiểm tra xem nó alert trước hay sau app caller.
    #    Nếu app caller alert TRƯỚC → app là culprit, không phải DB.
    top_candidate, top_score = sorted_candidates[0]
    is_store = any(st["name"] == top_candidate for st in stores)
    if is_store:
        callers = [edge["from"] for edge in services_data["edges"] if edge["to"] == top_candidate and edge["from"] in services]
        if callers:
            t_store = earliest_ts[top_candidate]
            t_callers = [earliest_ts[c] for c in callers if c in earliest_ts]
            if t_callers:
                t_caller_min = min(t_callers)
                if t_store > t_caller_min:
                    earliest_caller = min(callers, key=lambda c: earliest_ts[c])
                    caller_idx = -1
                    for idx, (name, val) in enumerate(sorted_candidates):
                        if name == earliest_caller:
                            caller_idx = idx
                            break
                    if caller_idx != -1:
                        temp = sorted_candidates.pop(caller_idx)
                        sorted_candidates.insert(0, (temp[0], top_score))
                        
    return sorted_candidates, combined_score, earliest_ts

print("Combined graph + temporal scorer helper function defined successfully.")

Combined graph + temporal scorer helper function defined successfully.


In [6]:
# Step 6: Implement Incident Cosine Similarity Retrieval & kNN Classifier
def retrieve_similar_incidents(cluster, query_text):
    query_vec = vectorizer.transform([query_text])
    sims = cosine_similarity(query_vec, tfidf_matrix).flatten()
    
    # Sort incidents by similarity descending, tie-breaking by incident date descending
    sorted_indices = []
    for idx, sim in enumerate(sims):
        if sim > 0:
            sorted_indices.append((idx, sim, pd.to_datetime(incident_list[idx]["ts"])))
    sorted_indices.sort(key=lambda x: (x[1], x[2]), reverse=True)
    
    top_incidents = []
    for idx, sim, _ in sorted_indices[:3]:
        top_incidents.append((incident_list[idx], sim))
        
    return top_incidents

def classify_root_cause(cluster, graph_top3, top_incidents, combined_score):
    services = cluster["services"]
    c_id = cluster["cluster_id"]
    
    if not top_incidents:
        root_cause = graph_top3[0][0]
        root_class = "other"
        confidence = float(graph_top3[0][1])
        actions = ["Investigate manually"]
        reasoning = "No similar historical incidents found. Fallback to top graph candidate."
        method = "graph-only-fallback"
    else:
        best_inc, best_score = top_incidents[0]
        root_cause = best_inc["root_cause_service"]
        if root_cause not in services:
            root_cause = graph_top3[0][0]
            
        root_class = best_inc["root_cause_class"]
        confidence = float(combined_score.get(root_cause, graph_top3[0][1]))
        
        # Parse remediation text into actions list
        remediation_str = best_inc["remediation"]
        parts = remediation_str.split(". ")
        actions = []
        for p in parts:
            p = p.strip()
            if p:
                if p.endswith('.'):
                    p = p[:-1].strip()
                if p:
                    actions.append(p)
                    
        reasoning = f"Matched historical incident {best_inc['id']} with TF-IDF similarity {best_score:.4f}."
        method = "graph+retrieval"
        
    return {
        "cluster_id": c_id,
        "graph_top3": [[name, round(float(val), 3)] for name, val in graph_top3],
        "root_cause": root_cause,
        "class": root_class,
        "confidence": round(confidence, 3),
        "actions": actions,
        "reasoning": reasoning,
        "similar_incidents": [item[0]["id"] for item in top_incidents],
        "method": method
    }

print("Incident retrieval and classification helper functions defined successfully.")

Incident retrieval and classification helper functions defined successfully.


In [7]:
# Step 7: Run analysis loop, classify root cause, write JSON output, and draw subgraphs
results = []
clusters = cluster_summary["clusters"]
multi_service_clusters = [c for c in clusters if len(c["services"]) > 1]

for cluster in multi_service_clusters:
    c_id = cluster["cluster_id"]
    services = cluster["services"]
    
    # Get alerts matching the cluster
    c_alerts = get_alerts_for_cluster(cluster, all_alerts)
    
    # 1. Run Graph + Temporal Scorer
    graph_candidates, combined_score, earliest_ts = compute_graph_temporal_candidates(cluster, c_alerts)
    graph_top3 = graph_candidates[:3]
    
    # 2. Build retrieval query and search past incidents
    services_str = " ".join(services)
    fps = [fp.replace("|", " ").replace("_", " ") for fp in cluster["fingerprints"]]
    fps_str = " ".join(fps)
    query_text = f"{services_str} {fps_str}"
    
    top_incidents = retrieve_similar_incidents(cluster, query_text)
    
    # 3. Classify root cause & actions
    rc_result = classify_root_cause(cluster, graph_top3, top_incidents, combined_score)
    results.append(rc_result)
    print(f"Processed cluster {c_id}:")
    print(f"  Graph Top-3: {graph_top3}")
    print(f"  Root Cause: {rc_result['root_cause']} (Class: {rc_result['class']}, Confidence: {rc_result['confidence']:.3f})")
    print(f"  Actions: {rc_result['actions']}")
    print()
    
    # 4. Visualize the cluster subgraph cleanly with causal path context
    G_undirected = G.to_undirected()
    context_nodes = set(services)
    for u in services:
        for v in services:
            if u != v:
                try:
                    for path in nx.all_simple_paths(G_undirected, u, v, cutoff=2):
                        context_nodes.update(path)
                except nx.NetworkXNoPath:
                    pass
    
    subg = G.subgraph(context_nodes)
    plt.figure(figsize=(7, 7))
    pos_sub = nx.circular_layout(subg)
    
    node_colors = []
    node_edge_colors = []
    for node in subg.nodes():
        if node == rc_result['root_cause']:
            node_colors.append('#ff2a2a')
            node_edge_colors.append('#2c3e50')
        elif node in services:
            node_colors.append('#54a0ff')
            node_edge_colors.append('#2c3e50')
        else:
            node_colors.append('#dcdde1')
            node_edge_colors.append('#7f8c8d')
            
    nx.draw_networkx_nodes(subg, pos_sub, node_color=node_colors, node_size=1000,
                           edgecolors=node_edge_colors, linewidths=1.5)
    
    edge_colors = []
    edge_widths = []
    for u, v in subg.edges():
        if u in services and v in services:
            edge_colors.append('#2c3e50')
            edge_widths.append(2.0)
        else:
            edge_colors.append('#7f8c8d')
            edge_widths.append(1.2)
            
    nx.draw_networkx_edges(subg, pos_sub, arrowstyle="->", arrowsize=15, 
                           edge_color=edge_colors, width=edge_widths, connectionstyle="arc3,rad=0.15")
    
    # Draw labels dynamically to prevent overlaps and clipping
    ax = plt.gca()
    for node, coords in pos_sub.items():
        x, y = coords
        factor = 1.22
        xl, yl = x * factor, y * factor
        ha = 'left' if x > 0.15 else ('right' if x < -0.15 else 'center')
        va = 'bottom' if y > 0.15 else ('top' if y < -0.15 else 'center')
        font_color = '#2c3e50' if node in services else '#7f8c8d'
        ax.text(xl, yl, node, fontsize=9, fontweight='bold', ha=ha, va=va, color=font_color)
        
    plt.title(f"Cluster {c_id} Subgraph\n(Red Node = Predicted Root Cause: {rc_result['root_cause']})", 
              fontsize=11, fontweight='bold', pad=15)
    
    ax.set_aspect('equal')
    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-1.5, 1.5)
    plt.axis('off')
    plt.show()
    print()

os.makedirs("./results", exist_ok=True)
output_data = {
    "clusters_analyzed": len(multi_service_clusters),
    "results": results
}
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(output_data, f, indent=2, ensure_ascii=False)

print(f"Analysis complete. Saved results for {len(multi_service_clusters)} clusters to {output_path}")

Processed cluster c-000-000:
  Graph Top-3: [('checkout-svc', 0.769), ('edge-lb', 0.728), ('payment-svc', 0.549)]
  Root Cause: payment-svc (Class: connection_pool_exhaustion, Confidence: 0.549)
  Actions: ['Rollback to v3.1', 'Scale pool 50 → 100 cushion', 'Add pool monitor alert > 80%']

Processed cluster c-004-000:
  Graph Top-3: [('search-svc', 1.0), ('checkout-svc', 0.6)]
  Root Cause: search-svc (Class: n_plus_1, Confidence: 1.000)
  Actions: ['Batch fetch related products', 'Add monitoring query count / request']

Processed cluster c-005-000:
  Graph Top-3: [('payment-svc', 1.0), ('notification-svc', 0.669), ('edge-lb', 0.6)]
  Root Cause: edge-lb (Class: ddos, Confidence: 0.600)
  Actions: ['WAF rate-limit + Cloudflare proxy', 'Geographic rule']

Analysis complete. Saved results for 3 clusters to ./results/rca_output.json
